
# Cleaning Dataset Biofarmaka — AgriDash Kabupaten Kediri

Notebook ini digunakan untuk melakukan **cleaning dan standardisasi dataset Biofarmaka BPS** untuk kebutuhan AgriDash.

## Aturan cleaning yang digunakan

- Tanda `-` pada dataset BPS dianggap **NULL / data tidak tersedia**, sehingga dikonversi menjadi `NaN`.
- Tanda `...` juga diperlakukan sebagai `NaN`.
- Nilai `0` yang memang tertulis sebagai angka `0` tetap dipertahankan sebagai **0**, karena berbeda dengan `-`.
- Baris `Kediri` pada dataset per kecamatan dianggap sebagai **total Kabupaten Kediri**, bukan kecamatan ke-27.
- `Kayen Kidul` dinormalisasi menjadi `Kayenkidul` agar konsisten dengan master kecamatan AgriDash.
- Data asli tingkat kecamatan **tidak dianggap sebagai estimasi**, sehingga `is_estimate = False`.
- Data luas panen tetap mempertahankan satuan dari sumber dan tidak dikonversi secara paksa.
- Hasil cleaning hanya disimpan sebagai **dataset clean**, sedangkan hasil validasi hanya ditampilkan di notebook dan **tidak disimpan sebagai CSV**.


In [ ]:

import os
import re
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)



## 1. Menentukan lokasi project

Path dibuat menggunakan `os` agar notebook dapat dijalankan dari repository GitHub tanpa menggunakan path Windows yang hardcode.


In [ ]:

def cari_project_root(start_path=None):
    start = os.path.abspath(start_path or os.getcwd())
    current = start

    while True:
        if (
            os.path.isdir(os.path.join(current, "data"))
            and (
                os.path.isdir(os.path.join(current, "notebooks"))
                or os.path.isdir(os.path.join(current, ".git"))
            )
        ):
            return current

        parent = os.path.dirname(current)

        if parent == current:
            break

        current = parent

    return os.path.abspath(os.path.join(start, ".."))


PROJECT_ROOT = cari_project_root()

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RAW_HORTI = os.path.join(DATA_DIR, "raw", "hortikultura")

# Output hasil cleaning
OUT_BIOFARMAKA = os.path.join(
    DATA_DIR,
    "clean",
    "hortikultura",
    "biofarmaka"
)

os.makedirs(OUT_BIOFARMAKA, exist_ok=True)

print("PROJECT ROOT :", PROJECT_ROOT)
print("RAW HORTI    :", RAW_HORTI)
print("CLEAN OUTPUT :", OUT_BIOFARMAKA)



## 2. Nama file sumber

Pastikan tiga file berikut berada di:

`data/raw/hortikultura/`


In [ ]:

FILE_LUAS_KAB = "Luas Panen Biofarmaka.csv"
FILE_LUAS_KEC = "Luas Panen Biofarmaka per kecamatan.csv"
FILE_PRODUKSI_KEC = "Produksi Biofarmaka per kecamatan.csv"

PATH_LUAS_KAB = os.path.join(RAW_HORTI, FILE_LUAS_KAB)
PATH_LUAS_KEC = os.path.join(RAW_HORTI, FILE_LUAS_KEC)
PATH_PRODUKSI_KEC = os.path.join(RAW_HORTI, FILE_PRODUKSI_KEC)

paths = {
    "Luas Panen Kabupaten": PATH_LUAS_KAB,
    "Luas Panen Kecamatan": PATH_LUAS_KEC,
    "Produksi Kecamatan": PATH_PRODUKSI_KEC
}

for nama, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"File {nama} tidak ditemukan:\n{path}"
        )

print("Semua file sumber ditemukan.")


In [ ]:

# Membaca dataset mentah

df_luas_kab_raw = pd.read_csv(
    PATH_LUAS_KAB,
    encoding="utf-8-sig"
)

df_luas_kec_raw = pd.read_csv(
    PATH_LUAS_KEC,
    encoding="utf-8-sig"
)

df_produksi_kec_raw = pd.read_csv(
    PATH_PRODUKSI_KEC,
    encoding="utf-8-sig"
)

print("Luas Panen Kabupaten :", df_luas_kab_raw.shape)
print("Luas Panen Kecamatan:", df_luas_kec_raw.shape)
print("Produksi Kecamatan  :", df_produksi_kec_raw.shape)



## 3. Melihat struktur data sebelum cleaning


In [ ]:

print("=== LUAS PANEN KABUPATEN ===")
display(df_luas_kab_raw.head())

print("\n=== LUAS PANEN PER KECAMATAN ===")
display(df_luas_kec_raw.head())

print("\n=== PRODUKSI PER KECAMATAN ===")
display(df_produksi_kec_raw.head())



## 4. Master kecamatan

Kabupaten Kediri memiliki 26 kecamatan. Baris `Kediri` yang muncul pada dataset BPS akan dipisahkan sebagai total Kabupaten.


In [ ]:

MASTER_KECAMATAN = [
    "Mojo",
    "Semen",
    "Ngadiluwih",
    "Kras",
    "Ringinrejo",
    "Kandat",
    "Wates",
    "Ngancar",
    "Plosoklaten",
    "Gurah",
    "Puncu",
    "Kepung",
    "Kandangan",
    "Pare",
    "Badas",
    "Kunjang",
    "Plemahan",
    "Purwoasri",
    "Papar",
    "Pagu",
    "Kayenkidul",
    "Gampengrejo",
    "Ngasem",
    "Banyakan",
    "Grogol",
    "Tarokan"
]

NORMALISASI_KECAMATAN = {
    "Kayen Kidul": "Kayenkidul"
}


def normalisasi_kecamatan(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    return NORMALISASI_KECAMATAN.get(x, x)


df_luas_kec_raw["Kecamatan"] = (
    df_luas_kec_raw["Kecamatan"]
    .map(normalisasi_kecamatan)
)

df_produksi_kec_raw["Kecamatan"] = (
    df_produksi_kec_raw["Kecamatan"]
    .map(normalisasi_kecamatan)
)


print("Jumlah master kecamatan:", len(MASTER_KECAMATAN))



## 5. Fungsi cleaning nilai

**Penting:** berdasarkan keterangan BPS yang digunakan pada dataset ini:

- `-` = **NULL**
- `...` = **NULL**
- angka `0` = tetap **0**

Jadi `-` **tidak boleh diubah menjadi 0** karena akan membuat data kosong dianggap sebagai produksi/luas panen nol.


In [ ]:

def bersihkan_nilai(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    # Kode data kosong dari sumber BPS
    if s in ["-", "..."]:
        return np.nan

    # Hilangkan pemisah ribuan jika ada
    s = s.replace(",", "")

    return pd.to_numeric(s, errors="coerce")



## 6. Cleaning luas panen per kecamatan

Format wide dari BPS diubah menjadi format long agar lebih mudah digunakan untuk database, analisis, dashboard, dan integrasi dengan dataset AgriDash lainnya.


In [ ]:

def ekstrak_nama_komoditas_luas(kolom):
    nama = re.sub(
        r"^Luas Panen\s+",
        "",
        str(kolom),
        flags=re.IGNORECASE
    )

    # Menghapus informasi satuan di dalam tanda kurung
    nama = re.sub(r"\s*\(.*$", "", nama).strip()

    return nama


def ekstrak_satuan_luas(kolom):
    lower = str(kolom).lower()

    if "pohon" in lower:
        return "Pohon"

    if "meter persegi" in lower or "(m2)" in lower:
        return "m2"

    return "Tidak diketahui"


def wide_ke_long_luas(df_raw):
    records = []

    for _, row in df_raw.iterrows():
        kecamatan = normalisasi_kecamatan(row["Kecamatan"])

        for kolom in df_raw.columns[1:]:
            records.append({
                "kecamatan": kecamatan,
                "komoditas": ekstrak_nama_komoditas_luas(kolom),
                "luas_panen": bersihkan_nilai(row[kolom]),
                "satuan_luas_panen": ekstrak_satuan_luas(kolom)
            })

    return pd.DataFrame(records)


df_luas_long = wide_ke_long_luas(df_luas_kec_raw)

# Data asli BPS tingkat kecamatan, bukan hasil estimasi
df_luas_long["is_estimate"] = False
df_luas_long["metode"] = "Data sumber BPS tingkat kecamatan"

# Status kualitas
df_luas_long["status_qc"] = np.where(
    df_luas_long["luas_panen"].isna(),
    "NULL",
    "OK"
)

display(df_luas_long.head(20))



## 7. Cleaning produksi per kecamatan


In [ ]:

def ekstrak_nama_komoditas_produksi(kolom):
    nama = re.sub(
        r"^Produksi\s+",
        "",
        str(kolom),
        flags=re.IGNORECASE
    )

    nama = re.sub(r"\s*\(.*$", "", nama).strip()

    return nama


def wide_ke_long_produksi(df_raw):
    records = []

    for _, row in df_raw.iterrows():
        kecamatan = normalisasi_kecamatan(row["Kecamatan"])

        for kolom in df_raw.columns[1:]:
            records.append({
                "kecamatan": kecamatan,
                "komoditas": ekstrak_nama_komoditas_produksi(kolom),
                "produksi_kg": bersihkan_nilai(row[kolom])
            })

    return pd.DataFrame(records)


df_produksi_long = wide_ke_long_produksi(
    df_produksi_kec_raw
)

df_produksi_long["is_estimate"] = False
df_produksi_long["metode"] = "Data sumber BPS tingkat kecamatan"

df_produksi_long["status_qc"] = np.where(
    df_produksi_long["produksi_kg"].isna(),
    "NULL",
    "OK"
)

display(df_produksi_long.head(20))



## 8. Memisahkan 26 kecamatan dan total Kabupaten

Baris `Kediri` tidak dimasukkan ke dataset kecamatan karena merupakan total Kabupaten.


In [ ]:

# LUAS PANEN
df_luas_kec_clean = df_luas_long[
    df_luas_long["kecamatan"] != "Kediri"
].copy()

df_luas_kab_clean = df_luas_long[
    df_luas_long["kecamatan"] == "Kediri"
].copy()


# PRODUKSI
df_produksi_kec_clean = df_produksi_long[
    df_produksi_long["kecamatan"] != "Kediri"
].copy()

df_produksi_kab_clean = df_produksi_long[
    df_produksi_long["kecamatan"] == "Kediri"
].copy()


print("Jumlah kecamatan pada luas panen:",
      df_luas_kec_clean["kecamatan"].nunique())

print("Jumlah kecamatan pada produksi:",
      df_produksi_kec_clean["kecamatan"].nunique())



## 9. Validasi hasil cleaning

Validasi **hanya ditampilkan di notebook** dan tidak disimpan sebagai file.

Pengecekan meliputi:
- apakah terdapat 26 kecamatan;
- apakah ada kecamatan yang hilang;
- apakah ada duplikasi kecamatan-komoditas;
- jumlah NULL;
- nilai negatif;
- perbandingan total 26 kecamatan dengan total Kabupaten dari BPS.

Perbedaan total **tidak otomatis berarti cleaning salah**, terutama jika sumber memiliki NULL atau terdapat perbedaan cakupan data.


In [ ]:

def tampilkan_validasi_kecamatan(df, nama):
    daftar_data = set(
        df["kecamatan"].dropna().unique()
    )

    master = set(MASTER_KECAMATAN)

    print(f"=== VALIDASI {nama} ===")
    print("Jumlah kecamatan :", len(daftar_data))

    print(
        "Kecamatan yang hilang:",
        sorted(master - daftar_data)
    )

    print(
        "Nama kecamatan di luar master:",
        sorted(daftar_data - master)
    )

    duplikat = df.duplicated(
        subset=["kecamatan", "komoditas"]
    ).sum()

    print(
        "Duplikasi kecamatan-komoditas:",
        duplikat
    )

    print(
        "Jumlah NULL:",
        df["luas_panen"].isna().sum()
        if "luas_panen" in df.columns
        else df["produksi_kg"].isna().sum()
    )


tampilkan_validasi_kecamatan(
    df_luas_kec_clean,
    "LUAS PANEN"
)

print()

tampilkan_validasi_kecamatan(
    df_produksi_kec_clean,
    "PRODUKSI"
)


In [ ]:

def validasi_total(df_kec, df_kab, kolom_nilai, nama_nilai):
    hasil = []

    komoditas_semua = sorted(
        set(df_kec["komoditas"].dropna())
        |
        set(df_kab["komoditas"].dropna())
    )

    for komoditas in komoditas_semua:

        nilai_kec = df_kec.loc[
            df_kec["komoditas"] == komoditas,
            kolom_nilai
        ]

        nilai_kab = df_kab.loc[
            df_kab["komoditas"] == komoditas,
            kolom_nilai
        ]

        total_kec = nilai_kec.sum(min_count=1)

        total_kab = (
            nilai_kab.iloc[0]
            if len(nilai_kab) > 0
            else np.nan
        )

        if pd.isna(total_kec) or pd.isna(total_kab):
            status = "TIDAK DAPAT DIVALIDASI"
            selisih = np.nan

        else:
            selisih = total_kec - total_kab

            status = (
                "SESUAI"
                if np.isclose(
                    total_kec,
                    total_kab,
                    atol=0.01
                )
                else "BERBEDA"
            )

        hasil.append({
            "komoditas": komoditas,
            "total_26_kecamatan": total_kec,
            "total_kabupaten_bps": total_kab,
            "selisih": selisih,
            "status": status
        })

    return pd.DataFrame(hasil)


validasi_luas = validasi_total(
    df_luas_kec_clean,
    df_luas_kab_clean,
    "luas_panen",
    "luas_panen"
)

validasi_produksi = validasi_total(
    df_produksi_kec_clean,
    df_produksi_kab_clean,
    "produksi_kg",
    "produksi_kg"
)

print("=== VALIDASI TOTAL LUAS PANEN ===")
display(validasi_luas)

print("=== VALIDASI TOTAL PRODUKSI ===")
display(validasi_produksi)



## 10. Pemeriksaan nilai NULL

Karena `-` dari BPS berarti NULL, bagian ini memperlihatkan komoditas dan kecamatan yang memiliki data kosong.

**NULL tidak diganti dengan 0.**


In [ ]:

print("=== NULL PADA LUAS PANEN ===")

display(
    df_luas_kec_clean[
        df_luas_kec_clean["luas_panen"].isna()
    ].sort_values(
        ["kecamatan", "komoditas"]
    )
)

print("=== NULL PADA PRODUKSI ===")

display(
    df_produksi_kec_clean[
        df_produksi_kec_clean["produksi_kg"].isna()
    ].sort_values(
        ["kecamatan", "komoditas"]
    )
)



## 11. Pemeriksaan satuan luas

Satuan tidak dikonversi secara otomatis. Ini penting karena dataset biofarmaka dapat memiliki komoditas yang dilaporkan dalam `m2` maupun `Pohon`.

Jika terdapat ketidaksesuaian satuan pada sumber, data ditandai untuk pemeriksaan lebih lanjut.


In [ ]:

print("Distribusi satuan luas panen:")
display(
    df_luas_kec_clean[
        ["komoditas", "satuan_luas_panen"]
    ]
    .drop_duplicates()
    .sort_values("komoditas")
)



## 12. Menyiapkan dataset final

Dataset final yang disimpan hanya dataset hasil cleaning. **File validasi tidak disimpan.**

Output:
- `luas_panen_biofarmaka_per_kecamatan.csv`
- `luas_panen_biofarmaka_kabupaten.csv`
- `produksi_biofarmaka_per_kecamatan.csv`
- `produksi_biofarmaka_kabupaten.csv`


In [ ]:

# Pembulatan angka
for df in [
    df_luas_kec_clean,
    df_luas_kab_clean
]:
    df["luas_panen"] = df["luas_panen"].round(2)


for df in [
    df_produksi_kec_clean,
    df_produksi_kab_clean
]:
    df["produksi_kg"] = df["produksi_kg"].round(2)


# Pastikan kolom berada pada urutan yang rapi
df_luas_kec_clean = df_luas_kec_clean[
    [
        "kecamatan",
        "komoditas",
        "luas_panen",
        "satuan_luas_panen",
        "is_estimate",
        "metode",
        "status_qc"
    ]
]

df_luas_kab_clean = df_luas_kab_clean[
    [
        "kecamatan",
        "komoditas",
        "luas_panen",
        "satuan_luas_panen",
        "is_estimate",
        "metode",
        "status_qc"
    ]
]

df_produksi_kec_clean = df_produksi_kec_clean[
    [
        "kecamatan",
        "komoditas",
        "produksi_kg",
        "is_estimate",
        "metode",
        "status_qc"
    ]
]

df_produksi_kab_clean = df_produksi_kab_clean[
    [
        "kecamatan",
        "komoditas",
        "produksi_kg",
        "is_estimate",
        "metode",
        "status_qc"
    ]
]


In [ ]:

# Nama file output
OUT_LUAS_KEC = os.path.join(
    OUT_BIOFARMAKA,
    "luas_panen_biofarmaka_per_kecamatan.csv"
)

OUT_LUAS_KAB = os.path.join(
    OUT_BIOFARMAKA,
    "luas_panen_biofarmaka_kabupaten.csv"
)

OUT_PROD_KEC = os.path.join(
    OUT_BIOFARMAKA,
    "produksi_biofarmaka_per_kecamatan.csv"
)

OUT_PROD_KAB = os.path.join(
    OUT_BIOFARMAKA,
    "produksi_biofarmaka_kabupaten.csv"
)


# Simpan dataset clean
df_luas_kec_clean.to_csv(
    OUT_LUAS_KEC,
    index=False,
    encoding="utf-8-sig"
)

df_luas_kab_clean.to_csv(
    OUT_LUAS_KAB,
    index=False,
    encoding="utf-8-sig"
)

df_produksi_kec_clean.to_csv(
    OUT_PROD_KEC,
    index=False,
    encoding="utf-8-sig"
)

df_produksi_kab_clean.to_csv(
    OUT_PROD_KAB,
    index=False,
    encoding="utf-8-sig"
)


print("Dataset clean berhasil disimpan ke:")
print(OUT_BIOFARMAKA)

print("\nFile output:")
print("-", os.path.basename(OUT_LUAS_KEC))
print("-", os.path.basename(OUT_LUAS_KAB))
print("-", os.path.basename(OUT_PROD_KEC))
print("-", os.path.basename(OUT_PROD_KAB))



## 13. Preview dataset final


In [ ]:

print("=== LUAS PANEN PER KECAMATAN ===")
display(df_luas_kec_clean.head(20))

print("=== PRODUKSI PER KECAMATAN ===")
display(df_produksi_kec_clean.head(20))

print("=== LUAS PANEN KABUPATEN ===")
display(df_luas_kab_clean)

print("=== PRODUKSI KABUPATEN ===")
display(df_produksi_kab_clean)



# Kesimpulan

Dataset Biofarmaka telah dibersihkan dan distandarkan untuk kebutuhan AgriDash. Nilai `-` dan `...` diperlakukan sebagai **NULL (`NaN`)**, sedangkan angka `0` tetap dipertahankan sebagai nilai nol. Data `Kediri` dipisahkan sebagai total Kabupaten dan tidak dianggap sebagai kecamatan ke-27. Hasil validasi hanya ditampilkan di notebook sebagai kontrol kualitas dan tidak disimpan sebagai file. Dataset hasil cleaning disimpan pada `data/clean/hortikultura/biofarmaka/`.
